##Read Silver Layer

##Daily Weather Summary

In [0]:
silver_df = (
    spark.read.parquet(
        "s3://weather-data-pipeline-235/silver/weather/"
    )
)
display(silver_df)

## Import Functions

In [0]:
from pyspark.sql.functions import (
    avg,
    max,
    min,
    round,
    weekofyear,
    month,
    year,
    sum,
    when,
    col
)


##common metrics

In [0]:
metrics = [
round(avg("temperature"),2)
        .alias("avg_temperature"),
round(max("temperature"),2)
        .alias("max_temperature"),
round(min("temperature"),2)
        .alias("min_temperature"),
round(avg("humidity"),2)
        .alias("avg_humidity"),
round(avg("wind_speed"),2)
        .alias("avg_wind_speed"),
sum(
        when(
            col("temperature_category")=="Hot",
            1
        ).otherwise(0)
    ).alias("hot_count"),
sum(
        when(
            col("temperature_category")=="Moderate",
            1
        ).otherwise(0)
    ).alias("moderate_count"),
sum(
        when(
            col("temperature_category")=="Cold",
            1
        ).otherwise(0)
    ).alias("cold_count"),
sum(
        when(
            col("humidity_category")=="Dry",
            1
        ).otherwise(0)
    ).alias("dry_count"),
sum(
        when(
            col("humidity_category")=="Normal",
            1
        ).otherwise(0)
    ).alias("normal_count"),
sum(
        when(
            col("humidity_category")=="Humid",
            1
        ).otherwise(0)
    ).alias("humid_count"),
sum(
        when(
            col("wind_category")=="Low",
            1
        ).otherwise(0)
    ).alias("low_wind_count"),
sum(
        when(
            col("wind_category")=="Medium",
            1
        ).otherwise(0)
    ).alias("medium_wind_count"),
sum(
        when(
            col("wind_category")=="High",
            1
        ).otherwise(0)
    ).alias("high_wind_count"),
sum(
        when(
            col("weather")=="Clouds",
            1
        ).otherwise(0)
    ).alias("clouds_count"),
sum(
        when(
            col("weather")=="Clear",
            1
        ).otherwise(0)
    ).alias("clear_count"),
sum(
        when(
            col("weather")=="Rain",
            1
        ).otherwise(0)
    ).alias("rain_count")
]


In [0]:
weather_daily_summary = (
    silver_df
    .groupBy(
        "city",
        "ingestion_date"
    )
    .agg(*metrics)
)


In [0]:
(
    weather_daily_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(
        "s3://weather-data-pipeline-235/gold/weather_daily_summary/"
    )
)

##Monthly Weather Summary

In [0]:
silver_month_df = (
    silver_df
    .withColumn(
        "year_value",
        year("ingestion_date")
    )
    .withColumn(
        "month_value",
        month("ingestion_date")
    )
)


weather_monthly_summary = (
    silver_month_df
    .groupBy(
        "city",
        "year_value",
        "month_value"
    )
    .agg(*metrics)
)

##Writing to Gold Layer

In [0]:
(
    weather_monthly_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(
        "s3://weather-data-pipeline-235/gold/weather_monthly_summary/"
    )
)

##Weekly weather summary

In [0]:
silver_week_df =(
    silver_df
    .withColumn(
        "week_number",
        weekofyear("ingestion_date")
    )
)

In [0]:
silver_week_df = (
    silver_df
    .withColumn(
        "week_number",
        weekofyear(
            "ingestion_date"
        )
    )
)
weather_weekly_summary = (
    silver_week_df
    .groupBy(
        "city",
        "week_number"
    )
    .agg(*metrics)
)



##Writing to gold layer

In [0]:
(
    weather_weekly_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(
        "s3://weather-data-pipeline-235/gold/weather_weekly_summary/"
    )
)

##Yearly Weather Summary

In [0]:
silver_year_df = (
    silver_df
    .withColumn(
        "year_value",
        year("ingestion_date")
    )
)
weather_yearly_summary = (
    silver_year_df
    .groupBy(
        "city",
        "year_value"
    )
    .agg(*metrics)
)

##writing to gold layer

In [0]:
(
    weather_yearly_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(
        "s3://weather-data-pipeline-235/gold/weather_yearly_summary/"
    )
)
